<a href="https://colab.research.google.com/github/F-Abir/GEE-Python-API/blob/main/DEM_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import os

# This will open a file upload dialog
uploaded = files.upload()

# Rename to exactly what the code expects
for filename in uploaded.keys():
    os.rename(filename, "/content/dem_bathy_PR.tif")
    print(f"File saved as: /content/dem_bathy_PR.tif")
    print(f"File size: {os.path.getsize('/content/dem_bathy_PR.tif')} bytes")

In [ ]:
# ==== 3D Bangladesh — Times New Roman + Blue→Cream→Light Red + Water Layer ====
# Input: /content/dem_bathy_PR.tif  (projected CRS in meters; land=+m, sea=−m)

# Improved dependency installation with version pinning
!pip -q install rasterio>=1.3.0 plotly>=5.14.0 numpy>=1.24.0 kaleido>=0.2.1

import os, numpy as np, rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio.transform import from_origin
import plotly.graph_objects as go

# ----------------- Paths & knobs -----------------
tif_path = "/content/dem_bathy_PR.tif"
assert os.path.exists(tif_path), "Upload dem_bathy_PR.tif to /content."

target_max_dim = 900  # downsample long side; 600–1200 is fine

# Plausible Z range (m) for BD + Bay of Bengal: clamps weird spikes
PLAUDIBLE_MIN, PLAUDIBLE_MAX = -2000.0, 2500.0

# Gentle vertical scale (apply BEFORE exaggeration)
land_geom_scale = 0.01     # scale elevations (>0 m)
sea_geom_scale  = 0.01       # scale depths (<0 m)
vertical_exaggeration = 0.50 # final multiplier (slightly reduced)

# Water layer settings
WATER_OPACITY = 0.25        # transparency of water layer
WATER_COLOR = '#00CED1'     # cyan/turquoise color for water

# ----------------- Read source -----------------
with rasterio.open(tif_path) as src:
    Zsrc = src.read(1, masked=True).astype("float64")
    tr   = src.transform
    crs  = src.crs
    Hs, Ws = Zsrc.shape
    b = src.bounds

# ----------------- North-up grid (no rotation/skew) -----------------
px = abs(tr.a) if tr.a != 0 else (b.right - b.left) / Ws
py = abs(tr.e) if tr.e != 0 else (b.top   - b.bottom) / Hs
dst_w = int(np.ceil((b.right - b.left) / px))
dst_h = int(np.ceil((b.top   - b.bottom) / py))
dst_tr = from_origin(b.left, b.top, px, py)   # b=d=0 → clean rectangle

Z = np.full((dst_h, dst_w), np.nan, dtype="float64")
reproject(
    source=Zsrc.filled(np.nan), destination=Z,
    src_transform=tr, src_crs=crs,
    dst_transform=dst_tr, dst_crs=crs,
    resampling=Resampling.bilinear, dst_nodata=np.nan
)

# ----------------- Optional downsample for speed -----------------
H, W = Z.shape
scale = min(target_max_dim / H, target_max_dim / W, 1.0)
if scale < 1.0:
    out_h, out_w = int(H * scale), int(W * scale)
    Z_small = np.full((out_h, out_w), np.nan, dtype="float64")
    reproject(
        source=Z, destination=Z_small,
        src_transform=dst_tr, src_crs=crs,
        dst_transform=from_origin(b.left, b.top, px/scale, py/scale),
        dst_crs=crs, resampling=Resampling.bilinear, dst_nodata=np.nan
    )
    Z = Z_small
    dst_tr = rasterio.Affine(dst_tr.a/scale, 0, dst_tr.c, 0, dst_tr.e/scale, dst_tr.f)
    H, W = Z.shape

# ----------------- Clean & clamp values -----------------
Z = np.where(np.isfinite(Z), Z, np.nan)
Z = np.clip(Z, PLAUDIBLE_MIN, PLAUDIBLE_MAX)

# COLOR field (legend in true meters; gentle robust clip within plausible box)
cmin, cmax = np.nanpercentile(Z, [5, 95])
if not (np.isfinite(cmin) and np.isfinite(cmax) and cmax > cmin):
    cmin, cmax = PLAUDIBLE_MIN, PLAUDIBLE_MAX
Z_color = np.clip(Z, cmin, cmax)

# GEOMETRY field (gentle scale, then mild exaggeration)
Z_geom = np.where(Z >= 0, Z * land_geom_scale, Z * sea_geom_scale)
Z_plot = Z_geom * vertical_exaggeration

# ----------------- Coordinates (km if projected) -----------------
cols, rows = np.meshgrid(np.arange(W), np.arange(H))
X = dst_tr.c + dst_tr.a * cols
Y = dst_tr.f + dst_tr.e * rows
is_proj = bool(crs and getattr(crs, "is_projected", False))
units_div = 1000.0 if is_proj else 1.0
units_name = "km" if is_proj else "map units"
Xk, Yk = X/units_div, Y/units_div

# ----------------- Blue→Cream (0 m)→Light-Red palette (soothing, no harsh reds) -----------------
def blue_cream_light_red(vmin, vmax):
    """
    Negative (sea): deep → mid → pale blue
    0 m pivot: warm cream
    Positive (land): light rose → light red (no strong reds)
    """
    if vmax == vmin:
        return [[0,'#ffffff'], [1,'#ffffff']]
    r0 = float(np.clip((0 - vmin) / (vmax - vmin), 0.0, 1.0))
    def npos(frac, side):  # map fractional stops to [0,r0] or [r0,1]
        return r0*frac if side == "neg" else r0 + (1 - r0)*frac

    # Sea blues
    deep_blue  = '#08306b'  # very deep blue
    mid_blue   = '#2c7fb8'  # mid blue
    pale_blue  = '#cfeaf6'  # shallow blue

    # Pivot at 0 m
    green_0m   = '#93C572'  # warm cream

    # Land light reds (soothing)
    rose1 = '#fde2e2'  # very light rose
    rose2 = '#f5c1c1'  # light rose
    redL  = '#ee8f8f'  # light red (top end, still soft)

    stops = [
        [max(0.0, npos(0.00,'neg')), deep_blue ],
        [max(0.0, npos(0.55,'neg')), mid_blue  ],
        [max(0.0, npos(0.92,'neg')), pale_blue ],
        [r0, green_0m],
        [min(1.0, npos(0.35,'pos')), rose1],
        [min(1.0, npos(0.70,'pos')), rose2],
        [1.0,                       redL ],
    ]
    out, last = [], -1
    for p,c in sorted(stops, key=lambda x:x[0]):
        if p > last:
            out.append([p,c]); last = p
    return out

colorscale = blue_cream_light_red(cmin, cmax)

# ----------------- Create water mask (where elevation <= 0) -----------------
water_mask = Z_plot <= 0

# Create water surface at 0 elevation (only where there's water)
Z_water = np.where(water_mask, 0.0, np.nan)

# ----------------- Plotly 3D surface -----------------
surf = go.Surface(
    x=Xk, y=Yk, z=Z_plot,
    surfacecolor=Z_color,
    colorscale=colorscale,
    cmin=cmin, cmax=cmax,
    showscale=True,
    colorbar=dict(
        title="Elevation (m)",
        tickfont=dict(family="Times New Roman", size=12),
        titlefont=dict(family="Times New Roman", size=14)
    ),
    contours=dict(
        z=dict(
            show=True, usecolormap=True,
            start=float(np.nanpercentile(Z_plot, 10)),
            end=float(np.nanpercentile(Z_plot, 90)),
            size=(cmax - cmin) * vertical_exaggeration / 30.0,
            project=dict(z=True)
        )
    ),
    name="Terrain"
)

# Water layer at sea level (0m) - only covers areas below sea level
water = go.Surface(
    x=Xk, y=Yk, z=Z_water,
    showscale=False,
    opacity=WATER_OPACITY,
    colorscale=[[0, WATER_COLOR], [1, WATER_COLOR]],
    name="Water Surface",
    hoverinfo='skip'
)

# Semi-transparent reference plane at 0
plane = go.Surface(
    x=Xk, y=Yk, z=np.zeros_like(Z_plot),
    showscale=False, opacity=0.08,
    colorscale=[[0,'#000000'],[1,'#000000']],
    name="Sea Level Reference",
    hoverinfo='skip'
)

fig = go.Figure(data=[surf, water, plane])
fig.update_traces(
    lightposition=dict(x=0.8, y=0.8, z=1.0),
    lighting=dict(ambient=0.55, diffuse=0.68, specular=0.20, roughness=0.95, fresnel=0.18)
)

fig.update_scenes(aspectmode='data')
fig.update_layout(
    title="3D Bangladesh Terrain & Bathymetry with Water Layer",
    width=1100, height=820,
    font=dict(family="Times New Roman", size=14),
    scene=dict(
        xaxis_title=f"X ({units_name})",
        yaxis_title=f"Y ({units_name})",
        zaxis_title=f"Elevation (m), exag×{vertical_exaggeration:g}",
        xaxis=dict(titlefont=dict(family="Times New Roman"), tickfont=dict(family="Times New Roman")),
        yaxis=dict(titlefont=dict(family="Times New Roman"), tickfont=dict(family="Times New Roman")),
        zaxis=dict(titlefont=dict(family="Times New Roman"), tickfont=dict(family="Times New Roman"),
                   zeroline=True, zerolinecolor='#aaa', gridcolor='#ddd'),
        camera=dict(eye=dict(x=1.55, y=1.55, z=1.0))
    )
)

fig.show()

# ----------------- Save files (PNG for LinkedIn + interactive HTML) -----------------
png_path_wide  = "/content/bd_3d_water.png"                 # wide image
png_path_square = "/content/bd_3d_water_square.png"         # square (great for LinkedIn)
html_path = "/content/bd_3d_water_interactive.html"

try:
    # Wide 4:3 PNG
    fig.write_image(png_path_wide,   width=1600, height=1200, scale=2)  # high-res PNG
    print(f"✓ Saved wide PNG: {png_path_wide}")

    # Square PNG (nice for LinkedIn feed)
    fig.write_image(png_path_square, width=2048, height=2048, scale=2)
    print(f"✓ Saved square PNG: {png_path_square}")
except Exception as e:
    print(f"⚠ PNG export failed (kaleido issue): {e}")
    print("  Interactive HTML will still be saved.")

# Interactive HTML (always works)
fig.write_html(html_path, include_plotlyjs="cdn")
print(f"✓ Saved interactive HTML: {html_path}")

# OPTIONAL: one-click download in Colab
try:
    from google.colab import files
    if os.path.exists(png_path_wide):
        files.download(png_path_wide)
    if os.path.exists(png_path_square):
        files.download(png_path_square)
    files.download(html_path)
    print("\n✓ Files downloaded successfully!")
except ImportError:
    print("\n→ Not running in Colab. Files saved to /content directory.")
except Exception as e:
    print(f"\n→ Download skipped: {e}")